# Liquid-Cooled Heatsink for TO-247 Packages

Reproduces the 2D cross-section simulation from Waffler §4.4 (ETH Zurich Diss. 2011).  
Full junction-to-coolant layer stack: Si die → Cu baseplate → TIM → Al cooler block with drilled (circular) or milled (rectangular) channels.

**Requires a running FEMM server** (`start_femm_server.bat` or `python -m py2femm_server --port 8082`).

In [ ]:
import sys
from pathlib import Path

# Make the example package importable from the notebook
repo_root = Path().resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from examples.heatflow.liquid_cooler_to247.config import (
    default_waffler_config, compute_h,
)
from examples.heatflow.liquid_cooler_to247.circular import build_circular
from examples.heatflow.liquid_cooler_to247.rectangular import build_rectangular
from examples.heatflow.liquid_cooler_to247.sweep import (
    parse_csv_result, compute_coupling_matrix, run_sweep,
)
from py2femm.client import FemmClient

client = FemmClient()

def run_fn(problem):
    result = client.run("\n".join(problem.lua_script))
    if result.error:
        raise RuntimeError(f"FEMM error: {result.error}")
    return result.csv_data or ""

print("Client ready.")

## 1 — Config & convective coefficient

In [ ]:
cfg = default_waffler_config(n_devices=3, p_loss=30.0)

h_circ = compute_h(cfg)
dh_rect = 2 * cfg.ch_w * cfg.ch_h / (cfg.ch_w + cfg.ch_h)
h_rect = compute_h(cfg, dh_mm=dh_rect, area_mm2=cfg.ch_w * cfg.ch_h)

print(f"n_devices      : {cfg.n_devices}")
print(f"cooler width   : {cfg.b_cp:.1f} mm")
print(f"n_channels     : {cfg.n_channels}")
print(f"h circular     : {h_circ:.0f} W/m²K  (Waffler ref: 9436 W/m²K)")
print(f"h rectangular  : {h_rect:.0f} W/m²K  (dh={dh_rect:.2f} mm)")

## 2 — Single run: circular vs rectangular

In [ ]:
raw_circ = run_fn(build_circular(cfg))
raw_rect = run_fn(build_rectangular(cfg))

res_circ = parse_csv_result(raw_circ, cfg.n_devices)
res_rect = parse_csv_result(raw_rect, cfg.n_devices)

print("Circular channels:")
for k, v in sorted(res_circ.items()):
    print(f"  {k} = {v:.2f}")

print("\nRectangular channels:")
for k, v in sorted(res_rect.items()):
    print(f"  {k} = {v:.2f}")

## 3 — Junction & case temperatures bar chart

In [ ]:
n = cfg.n_devices
devices = [f"D{i}" for i in range(n)]
x = np.arange(n)
w = 0.35

T_j_circ   = [res_circ[f"T_j_{i}"]    for i in range(n)]
T_j_rect   = [res_rect[f"T_j_{i}"]    for i in range(n)]
T_case_circ = [res_circ[f"T_case_{i}"] for i in range(n)]
T_case_rect = [res_rect[f"T_case_{i}"] for i in range(n)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, T_circ, T_rect, ylabel, title in [
    (axes[0], T_j_circ,   T_j_rect,   "T_j [K]",    "Junction temperature"),
    (axes[1], T_case_circ, T_case_rect, "T_case [K]", "Case temperature"),
]:
    ax.bar(x - w/2, T_circ, w, label="Circular",    color="#1565c0", alpha=0.85)
    ax.bar(x + w/2, T_rect, w, label="Rectangular", color="#2e7d32", alpha=0.85)
    ax.axhline(cfg.t_inlet, color="red", lw=1.2, ls="--", label=f"T_inlet={cfg.t_inlet:.0f} K")
    ax.set_xticks(x)
    ax.set_xticklabels(devices)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

fig.suptitle(f"Waffler defaults — {n} TO-247 devices @ {cfg.devices[0].p_loss:.0f} W each", fontsize=11)
plt.tight_layout()
plt.show()

## 4 — Thermal coupling matrix

In [ ]:
C_circ = compute_coupling_matrix(cfg, builder="circular",    run_fn=run_fn)
C_rect = compute_coupling_matrix(cfg, builder="rectangular", run_fn=run_fn)

print("Coupling matrix — circular [K/W]:")
print(np.round(C_circ, 4))
print("\nCoupling matrix — rectangular [K/W]:")
print(np.round(C_rect, 4))

In [ ]:
def plot_coupling_matrix(C, title, ax):
    im = ax.imshow(C, cmap="YlOrRd", aspect="equal")
    plt.colorbar(im, ax=ax, label="K/W")
    n = C.shape[0]
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{C[i,j]:.4f}", ha="center", va="center", fontsize=9,
                    color="white" if C[i,j] > C.max() * 0.6 else "black")
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels([f"source D{k}" for k in range(n)], rotation=30, ha="right")
    ax.set_yticklabels([f"sensor D{i}" for i in range(n)])
    ax.set_title(title)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
plot_coupling_matrix(C_circ, "Circular channels — C [K/W]", ax1)
plot_coupling_matrix(C_rect, "Rectangular channels — C [K/W]", ax2)
fig.suptitle("Thermal coupling matrix  C[source, sensor] = (T_j[sensor] − T_inlet) / P_source",
             fontsize=10)
plt.tight_layout()
plt.show()

## 5 — Parametric sweep: R_th vs power level

In [ ]:
import io, csv

p_loss_values = [10.0, 20.0, 30.0, 40.0, 50.0]

buf = io.StringIO()
run_sweep(
    cfg=cfg,
    builders=["circular", "rectangular"],
    p_loss_values=p_loss_values,
    run_fn=run_fn,
    out=buf,
)
buf.seek(0)
rows = list(csv.DictReader(buf))
print(f"{len(rows)} rows written")
rows[:2]

In [ ]:
# Plot R_th,j-inlet for device 0 vs power for both builders
def get_series(rows, builder, key):
    return [
        (float(r["p_loss"]), float(r[key]))
        for r in rows if r["builder"] == builder and r[key]
    ]

fig, ax = plt.subplots(figsize=(7, 4))

for builder, color, marker in [("circular", "#1565c0", "o"), ("rectangular", "#2e7d32", "s")]:
    pts = get_series(rows, builder, "Rth_j_inlet_0")
    if pts:
        p_vals, rth_vals = zip(*pts)
        ax.plot(p_vals, rth_vals, marker=marker, color=color, label=f"{builder} — D0")

ax.set_xlabel("P_loss per device [W]")
ax.set_ylabel("R_th,j-inlet [K/W]")
ax.set_title("Junction-to-inlet thermal resistance vs. power (device 0)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Waffler §4.4 validation (single device, circular)

Target from Waffler Table 4.17: **ΔT_h-i ≈ 4.55 K** (Al block + convection, 30 W, h ≈ 9436 W/m²K)

In [ ]:
cfg1 = default_waffler_config(n_devices=1, p_loss=30.0)
raw1 = run_fn(build_circular(cfg1))
res1 = parse_csv_result(raw1, cfg1.n_devices)

T_h  = res1["T_h_surface"]
T_j  = res1["T_j_0"]
dT_cooler = T_h - cfg1.t_inlet
dT_stack  = T_j - cfg1.t_inlet
Rth_cooler = dT_cooler / cfg1.devices[0].p_loss
Rth_stack  = dT_stack  / cfg1.devices[0].p_loss

print(f"T_inlet        = {cfg1.t_inlet:.2f} K  ({cfg1.t_inlet - 273.15:.1f} °C)")
print(f"T_h_surface    = {T_h:.2f} K")
print(f"T_j            = {T_j:.2f} K")
print(f"ΔT_cooler      = {dT_cooler:.3f} K  (Waffler target: 4.55 K)")
print(f"ΔT_full stack  = {dT_stack:.3f} K")
print(f"R_th,cooler    = {Rth_cooler:.4f} K/W")
print(f"R_th,j-inlet   = {Rth_stack:.4f} K/W")

ok = 4.1 <= dT_cooler <= 5.0
print(f"\nValidation: {'✓ PASS' if ok else '✗ FAIL'} (target 4.1–5.0 K)")